In [0]:
%sql
USE CATALOG adb_dream_analytics_prod;
CREATE SCHEMA IF NOT EXISTS gold MANAGED LOCATION 'abfss://gold@adlsgen2dreamprod.dfs.core.windows.net/dream_app/';
USE SCHEMA gold;

In [0]:
%sql
CREATE TABLE IF NOT EXISTS gold.dim_drivers (
    driver_sk BIGINT GENERATED ALWAYS AS IDENTITY,
    driver_id INT,
    first_name STRING,
    last_name STRING,
    full_name STRING,
    phone_number STRING,
    driver_rating DECIMAL(3,2),
    city STRING,
    last_updated_timestamp TIMESTAMP,
    start_date DATE,
    end_date DATE,
    is_current BOOLEAN,
    scd2_hash STRING,
    _processed_at TIMESTAMP
) USING DELTA;

CREATE OR REPLACE TEMPORARY VIEW v_drivers_source AS 
SELECT
    driver_id ,
    first_name,
    last_name,
    concat_ws(' ', first_name , last_name) AS full_name ,
    driver_rating ,
    phone_number ,
    city ,
    try_cast(last_updated_timestamp AS TIMESTAMP) AS last_updated_timestamp,
    md5(upper(trim(city))) AS source_hash,
    current_timestamp() AS _processed_at
FROM adb_dream_analytics_prod.silver.drivers;

MERGE INTO dim_drivers AS Target
USING (
    SELECT
        s.driver_id AS merge_key,
        s.*
    FROM v_drivers_source s
    JOIN dim_drivers t ON t.driver_id = s.driver_id
    WHERE t.is_current = true AND(
        t.scd2_hash <> s.source_hash
        OR t.first_name IS DISTINCT FROM s.first_name
        OR t.last_name IS DISTINCT FROM s.last_name
        OR t.driver_rating IS DISTINCT FROM s.driver_rating
        OR t.phone_number IS DISTINCT FROM s.phone_number
    )

    UNION ALL

    SELECT
        NULL AS merge_key,
        s.*
    FROM v_drivers_source s
    LEFT JOIN dim_drivers t ON t.driver_id = s.driver_id
        AND t.is_current = true
        AND t.scd2_hash = s.source_hash
    WHERE t.is_current IS NULL
) AS Source
ON Target.driver_id = Source.merge_key
    AND Target.is_current = true

WHEN MATCHED AND target.scd2_hash <> source.source_hash THEN
    UPDATE SET
        Target.end_date =  current_date(),
        Target.is_current = false
        
WHEN MATCHED AND target.scd2_hash = source.source_hash AND (
    target.first_name IS DISTINCT FROM source.first_name
    OR target.last_name IS DISTINCT FROM source.last_name
    OR target.driver_rating IS DISTINCT FROM source.driver_rating
    OR target.phone_number IS DISTINCT FROM source.phone_number
) THEN
    UPDATE SET
        Target.first_name = Source.first_name,
        Target.last_name = Source.last_name,
        target.full_name = Source.full_name,
        Target.driver_rating = Source.driver_rating,
        Target.phone_number = Source.phone_number,
        Target.last_updated_timestamp = Source.last_updated_timestamp,
        target._processed_at = Source._processed_at

WHEN NOT MATCHED THEN
    INSERT (
        driver_id, first_name, last_name, full_name, driver_rating, phone_number, city, last_updated_timestamp, start_date,
        end_date, is_current, scd2_hash, _processed_at
    )
    VALUES (
        Source.driver_id, Source.first_name, Source.last_name, Source.full_name, Source.driver_rating, Source.phone_number,
        Source.city, Source.last_updated_timestamp, current_date(), NULL, true, Source.source_hash, source._processed_at
    );